In [8]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input

# Load the dataset
data = pd.read_csv('AIDS_Classification_50000.csv')

# Separate features (X) and target (y)
X = data.drop(columns=['infected'])  # Assuming 'infected' is the target column
y = data['infected']

# Initialize k-fold cross-validation
k = 10  # Number of folds
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# Metrics storage
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
roc_auc_scores = []

# Function to build the dense neural network model
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),  # Define the input shape here
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# k-Fold Cross-Validation
fold = 1
for train_index, test_index in kf.split(X):
    print(f"Fold {fold}:")
    
    # Split data into train and test sets for this fold
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build and train the model
    model = build_model(X_train_scaled.shape[1])
    model.fit(X_train_scaled, y_train, 
              epochs=20,  # Reduce epochs for cross-validation efficiency
              batch_size=64,  # Adjust batch size for faster training
              verbose=0)

    # Evaluate the model
    y_pred_prob = model.predict(X_test_scaled).flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_prob)

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    roc_auc_scores.append(roc_auc)

    # Print fold metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}\n")
    
    fold += 1

# Aggregate results
print("k-Fold Cross-Validation Results (Mean ± Std Dev):")
print(f"Accuracy: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
print(f"Precision: {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"Recall: {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
print(f"F1-Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"ROC-AUC: {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")


Fold 1:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7092
  Precision: 0.5901
  Recall: 0.1788
  F1-Score: 0.2745
  ROC-AUC: 0.7030

Fold 2:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7014
  Precision: 0.5421
  Recall: 0.1923
  F1-Score: 0.2839
  ROC-AUC: 0.6899

Fold 3:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.6974
  Precision: 0.5556
  Recall: 0.2451
  F1-Score: 0.3402
  ROC-AUC: 0.6948

Fold 4:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7112
  Precision: 0.6216
  Recall: 0.1357
  F1-Score: 0.2228
  ROC-AUC: 0.6988

Fold 5:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7022
  Precision: 0.5558
  Recall: 0.2023
  F1-Score: 0.2966
  ROC-AUC: 0.6930

Fold 6:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
  Accuracy: 0.7122
  Precision: 0.6079
  Recall: 0.1796
  F1-Score: 0.2772
  ROC-AUC: 0.7079

Fold 7:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.6982
  Precision: 0.5761
  Recall: 0.2085
  F1-Score: 0.3062
  ROC-AUC: 0.7025


In [9]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input

# Load the dataset
data = pd.read_csv('AIDS_Classification_50000.csv')

# Separate features (X) and target (y)
X = data.drop(columns=['infected'])  # Assuming 'infected' is the target column
y = data['infected']

# Initialize k-fold cross-validation
k = 10  # Number of folds
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# Metrics storage
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
roc_auc_scores = []
y_test_all = []  # Collect all true labels
y_pred_all = []  # Collect all predicted labels

# Function to build the dense neural network model
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),  # Define the input shape here
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# k-Fold Cross-Validation
fold = 1
for train_index, test_index in kf.split(X):
    print(f"Fold {fold}:")
    
    # Split data into train and test sets for this fold
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build and train the model
    model = build_model(X_train_scaled.shape[1])
    model.fit(X_train_scaled, y_train, 
              epochs=20,  # Reduce epochs for cross-validation efficiency
              batch_size=64,  # Adjust batch size for faster training
              verbose=0)

    # Evaluate the model
    y_pred_prob = model.predict(X_test_scaled).flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

    # Collect predictions and true labels
    y_test_all.extend(y_test)  # Append true labels for the current fold
    y_pred_all.extend(y_pred)  # Append predicted labels for the current fold

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_prob)

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    roc_auc_scores.append(roc_auc)

    # Print fold metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}\n")
    
    fold += 1

# Aggregate results
print("k-Fold Cross-Validation Results (Mean ± Std Dev):")
print(f"Accuracy: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
print(f"Precision: {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"Recall: {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
print(f"F1-Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"ROC-AUC: {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")

# Final Confusion Matrix
final_cm = confusion_matrix(y_test_all, y_pred_all)
print("\nFinal Confusion Matrix Across All Folds:")
print(final_cm)


Fold 1:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7108
  Precision: 0.6144
  Recall: 0.1606
  F1-Score: 0.2546
  ROC-AUC: 0.7022

Fold 2:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7062
  Precision: 0.5748
  Recall: 0.1748
  F1-Score: 0.2681
  ROC-AUC: 0.6913

Fold 3:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.6962
  Precision: 0.5619
  Recall: 0.2055
  F1-Score: 0.3010
  ROC-AUC: 0.6946

Fold 4:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7104
  Precision: 0.5800
  Recall: 0.1830
  F1-Score: 0.2782
  ROC-AUC: 0.7009

Fold 5:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.6998
  Precision: 0.5517
  Recall: 0.1753
  F1-Score: 0.2660
  ROC-AUC: 0.6940

Fold 6:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7124
  Precision: 0.5939
  Recall: 0.2036
  F1-Score: 0.3033
  ROC-AUC: 0.7094

Fold 7:
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
  Accuracy: 0.7032
  Precision: 0.6184
  Recall: 0.1847
  F1-Score: 0.2845
  ROC-AUC: 0.7032
